# 🏠 EARLY BIRD — Prediksi Harga Rumah Depok
### Bengkel Koding Data Science | EDA → Direct Modeling → Preprocessing → Feature Selection → Hyperparameter Tuning → Deployment

**Dataset:** `Raw-Final-Rumah.csv` (file asli dari tugas). **Target:** `Harga` dalam rupiah (regresi).

**Protokol ilmiah:** Satu populasi listing yang dapat dibandingkan, satu pembagian data bersama (70% train, 15% validation, 15% test), dan **test set hanya dibuka satu kali** setelah semua pilihan model dilakukan menggunakan validation/CV training. Transformasi yang dipelajari dari data (`SimpleImputer`, `OneHotEncoder`, `QuantileTransformer`, selector) selalu berada dalam `Pipeline` dan hanya di-*fit* pada training.

**Penting:** Direct Modeling di sini adalah *baseline minimal*; model scikit-learn tidak menerima nilai harga `"2,5 Miliar"` atau teks `"LT : 50 m²"` mentah, sehingga decoding teknis minimal tetap diperlukan. Tahap preprocessing lanjutan memperbaiki kolom bergeser, nilai tidak masuk akal, dan menambah fitur turunan. Pemilihan rentang harga/kamar adalah batas studi **sementara**, bukan kebenaran universal.

## 0. Persiapan Colab dan input data

Unggah **file yang sama persis**: `Raw-Final-Rumah.csv` ke sidebar Files Colab, atau jalankan sel upload. Jika menjalankan di lingkungan lokal dengan file itu satu folder, notebook otomatis mendeteksinya. Semua grafik, tabel, metrik, model, dan berkas deployment disimpan ke folder `earlybird_outputs/`.

In [ ]:
# Jalankan sel ini terlebih dahulu (instalasi hanya jika perlu).
%pip -q install -U "scikit-learn>=1.3,<2" pandas numpy matplotlib seaborn joblib streamlit

In [ ]:
import os, re, json, warnings, shutil, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE=42
OUT=Path('earlybird_outputs'); OUT.mkdir(parents=True, exist_ok=True)

# Kosongkan None agar menggunakan data penuh. Angka kecil hanya untuk uji cepat saat debugging.
SMOKE_LIMIT=None
CANDIDATES=[Path('/content/Raw-Final-Rumah.csv'), Path('Raw-Final-Rumah.csv'), Path('/mnt/data/Raw-Final-Rumah.csv')]
DATA_PATH=next((p for p in CANDIDATES if p.exists()),None)
if DATA_PATH is None:
    try:
        from google.colab import files
        print('Unggah Raw-Final-Rumah.csv dari komputer:')
        uploaded=files.upload()
        DATA_PATH=Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError('Letakkan Raw-Final-Rumah.csv pada folder kerja.')
print('Dataset:',DATA_PATH)
raw=pd.read_csv(DATA_PATH, dtype=str)
if SMOKE_LIMIT is not None:
    raw=raw.sample(n=min(SMOKE_LIMIT,len(raw)),random_state=42).reset_index(drop=True)
assert {'Harga','Kamar Tidur','Kamar Mandi','Garasi','Luas Tanah','Luas Bangunan','Kecamatan','Page'}.issubset(raw.columns)
print('Ukuran:',raw.shape)
display(raw.head())

## 1. EDA — audit kualitas, pola data, dan insight

Kita tidak boleh menganggap semua `NaN` sebagai data hilang permanen. Pada sebagian listing, keterangan luas tanah (`LT`) atau luas bangunan (`LB`) tersisip di kolom Kamar Tidur/Mandi/Garasi akibat pola scraping yang berubah. `Page` adalah informasi pagination, bukan karakteristik rumah. Karena dataset tidak menyediakan ID listing unik, deduplikasi tanpa `Page` hanya mendekati deteksi listing berulang dan dapat menghapus rumah berbeda dengan atribut identik; keterbatasan ini dicatat.

In [ ]:
# Audit awal — **jangan** langsung drop semua baris ber-NaN.
columns=list(raw.columns)
issues=[
 ('Missing values: '+c, int(raw[c].isna().sum()), 'Field kosong pada hasil scraping')
 for c in columns
]
issues += [
 ('Duplikat persis semua kolom',int(raw.duplicated().sum()),'Bisa akibat pengulangan hasil scraping'),
 ('Duplikat atribut tanpa Page',int(raw.duplicated(subset=[c for c in columns if c!='Page']).sum()),'Listing mungkin muncul berulang di halaman lain'),
 ('Kamar Tidur berisi LT/LB', int(raw['Kamar Tidur'].fillna('').str.contains(r'^\s*(?:LT|LB)\s*:',flags=re.I,regex=True).sum()), 'Kolom bergeser / listing non-rumah'),
 ('Harga: Hubungi Agen', int(raw['Harga'].str.contains('Hubungi Agen',case=False,na=False).sum()), 'Harga tidak dipublikasikan'),
 ('Harga per luas (/m² atau /ha)',int(raw['Harga'].str.contains('/',regex=False,na=False).sum()),'Satuan harga tidak sebanding dengan total rumah'),
 ('Harga berlabel Triliun',int(raw['Harga'].str.contains('Triliun',case=False,na=False).sum()),'Dapat berupa properti komersial atau anomali'),
]
quality=pd.DataFrame(issues,columns=['Masalah','Jumlah baris','Kemungkinan penyebab'])
display(quality)
quality.to_csv(OUT/'01_audit_kualitas.csv',index=False)
print('Jumlah wilayah/lokasi mentah:',raw['Kecamatan'].nunique())
print('Distribusi contoh teks Harga:')
display(raw['Harga'].value_counts().head(10))

### 1.1 Parsing target dan penentuan *scope* studi

Harga dikenali secara eksplisit: `Ribu`, `Juta`, `Miliar`, `Triliun`, boleh berakhiran `Total`. Harga `Hubungi Agen` atau harga **per luas** tidak bisa langsung menjadi target total. Untuk studi **rumah tinggal** kita gunakan kandidat harga **Rp100 juta–Rp50 miliar** dan **1–15 kamar tidur**; batas ini sengaja dicatat agar bisa dievaluasi, karena listing kost, tanah, proyek besar dan salah tulis mempunyai distribusi berbeda. Baris di luar scope **tidak otomatis dianggap salah**. Dengan ketiadaan ID listing, kita hilangkan pola duplikat persis **tanpa Page** untuk mengurangi risiko train-test leakage.

In [ ]:
PRICE_PATTERN=re.compile(r'^\s*([0-9]+(?:[.,][0-9]+)?)\s*(Ribu|Juta|Miliar|Triliun)(?:\s+Total)?\s*$', re.I)
UNIT={'ribu':1_000,'juta':1_000_000,'miliar':1_000_000_000,'triliun':1_000_000_000_000}
def parse_price(x):
    if pd.isna(x): return np.nan
    match=PRICE_PATTERN.fullmatch(str(x).strip())
    if match is None: return np.nan
    return float(match.group(1).replace(',','.'))*UNIT[match.group(2).lower()]
def simple_number(x):
    if pd.isna(x):return np.nan
    m=re.fullmatch(r'\s*([0-9]+(?:[.,][0-9]+)?)\s*',str(x))
    return float(m.group(1).replace(',','.')) if m else np.nan
def area_number(s):
    if not s:return np.nan
    s=str(s).strip()
    # Tanda titik 1.200 pada luas dianggap pemisah ribuan; koma desimal.
    if ',' in s: s=s.replace('.','').replace(',','.')
    elif re.fullmatch(r'\d{1,3}(?:\.\d{3})+',s):s=s.replace('.','')
    try:return float(s)
    except ValueError:return np.nan

raw['harga_rp']=raw['Harga'].map(parse_price)
raw['kamar_numeric']=raw['Kamar Tidur'].map(simple_number)
scope=raw.loc[raw['harga_rp'].between(1e8,5e10) & raw['kamar_numeric'].between(1,15)].copy()
raw_scope_count=len(scope)
duplicate_count=int(scope.duplicated(subset=columns[:-1]).sum())
scope=scope.drop_duplicates(subset=columns[:-1],keep='first').reset_index(drop=True)
print(f'Total raw = {len(raw):,}; kandidat scope = {raw_scope_count:,}; pola duplikat di scope = {duplicate_count:,}; baris analisis = {len(scope):,}')
print('Proporsi cakupan:',round(len(scope)/len(raw)*100,2),'%')
assert scope['harga_rp'].notna().all() and (scope['harga_rp']>0).all()

### 1.2 Membaca ulang informasi yang bergeser

- **Baseline direct**: ambil data numerik hanya dari kolom yang sesuai (konversi minimal, tanpa recovery).
- **Preprocessing lengkap**: cari label `LT:` / `LB:` pada semua kolom data fisik, lalu tempatkan pada fitur yang benar; nilai numerik di luar batas plausibilitas dijadikan missing dan diimputasi **hanya dari train**.
- **Feature engineering**: tambahkan `total_luas` dan `rasio_lb_lt` jika kedua luas tersedia.

Batas input untuk rumah tinggal: kamar mandi ≤15, garasi ≤12, LT ≤20.000 m², LB ≤10.000 m². Nilai di luar batas ditandai sebagai anomali (tidak langsung diartikan pasti salah). Jangan menyatukan nilai `Page` ke fitur prediktor.

In [ ]:
AREA_TAG=re.compile(r'\b(LT|LB)\s*:\s*([0-9]+(?:[.,][0-9]+)*)',re.I)
PHYSICAL=['Kamar Tidur','Kamar Mandi','Garasi','Luas Tanah','Luas Bangunan']
NUMERIC_BASE=['kamar_tidur','kamar_mandi','garasi','luas_tanah','luas_bangunan']
NUMERIC_FULL=NUMERIC_BASE+['total_luas','rasio_lb_lt']
CAT=['lokasi']

def within(v,low,high):
    return float(v) if pd.notna(v) and low<=v<=high else np.nan

def get_area(row,tag):
    preferred=['Luas Tanah','Kamar Mandi','Garasi','Kamar Tidur','Luas Bangunan'] if tag=='LT' else ['Luas Bangunan','Garasi','Kamar Mandi','Luas Tanah','Kamar Tidur']
    for key in preferred:
        content=row.get(key)
        if pd.isna(content):continue
        for m in AREA_TAG.finditer(str(content)):
            if m.group(1).upper()==tag:
                return area_number(m.group(2))
    return np.nan

def parse_baseline(df):
    result=pd.DataFrame(index=df.index)
    result['kamar_tidur']=df['Kamar Tidur'].map(simple_number)
    result['kamar_mandi']=df['Kamar Mandi'].map(simple_number)
    result['garasi']=df['Garasi'].map(simple_number)
    result['luas_tanah']=df['Luas Tanah'].map(lambda s: area_number(m.group(1)) if pd.notna(s) and (m:=re.search(r'LT\s*:\s*([0-9.,]+)',str(s),re.I)) else np.nan)
    result['luas_bangunan']=df['Luas Bangunan'].map(lambda s: area_number(m.group(1)) if pd.notna(s) and (m:=re.search(r'LB\s*:\s*([0-9.,]+)',str(s),re.I)) else np.nan)
    # Konversi teknis minimum dan validasi domain agar input estimator terdefinisi.
    for key,hi in [('kamar_tidur',15),('kamar_mandi',15),('garasi',12),('luas_tanah',20000),('luas_bangunan',10000)]:
        result[key]=result[key].apply(lambda x:within(x,0 if key!='kamar_tidur' else 1,hi))
    result['lokasi']=df['Kecamatan'].fillna('Tidak diketahui').astype(str).str.strip()
    return result

def parse_full(df):
    result=parse_baseline(df)
    result['luas_tanah']=[within(get_area(row,'LT'),1,20000) for row in df.to_dict('records')]
    result['luas_bangunan']=[within(get_area(row,'LB'),1,10000) for row in df.to_dict('records')]
    result['total_luas']=result['luas_tanah']+result['luas_bangunan']
    result['rasio_lb_lt']=result['luas_bangunan']/result['luas_tanah']
    result['rasio_lb_lt']=result['rasio_lb_lt'].replace([np.inf,-np.inf],np.nan).clip(upper=20)
    return result

X_basic=parse_baseline(scope)
X_clean=parse_full(scope)
y=scope['harga_rp'].astype(float)
print('Missing value baseline vs full (sebelum imputasi)')
display(pd.DataFrame({'baseline':X_basic[NUMERIC_BASE].isna().sum(), 'full':X_clean[NUMERIC_BASE].isna().sum()}))
print('Tabel contoh pemulihan data:')
example=(X_basic['luas_tanah'].isna()&X_clean['luas_tanah'].notna())
display(pd.concat([scope.loc[example,['Kamar Mandi','Garasi','Luas Tanah','Luas Bangunan']].head(5),X_clean.loc[example,['luas_tanah','luas_bangunan']].head(5)],axis=1))

### 1.3 Visualisasi dan tiga insight yang terhubung dengan keputusan model

Insight harus berasal dari data, bukan klaim di luar dataset. Statistik `harga` berikut mencerminkan **harga penawaran**, bukan harga transaksi aktual. Lokasi bernama `Kecamatan` dapat berisi tingkat administratif berbeda (contoh: nama kawasan). Kita menyebutnya sebagai **label lokasi mentah** di model.

In [ ]:
from scipy.stats import spearmanr
sns.set_theme(style='whitegrid')
fig,ax=plt.subplots(1,3,figsize=(17,4.5))
sns.histplot(y/1e9, bins=65, ax=ax[0],color='#2563eb');ax[0].set(title='Distribusi harga rumah',xlabel='Harga (miliar Rp)',ylabel='Frekuensi')
sns.histplot(np.log10(y),bins=65,ax=ax[1],color='#14b8a6');ax[1].set(title='Distribusi log10 harga',xlabel='log10(Harga Rp)')
loc_count=X_clean.groupby('lokasi').size()
loc_median=pd.DataFrame({'n':loc_count,'median':pd.concat([X_clean['lokasi'],y],axis=1).groupby('lokasi')['harga_rp'].median() if 'harga_rp' in X_clean else scope.groupby('Kecamatan')['harga_rp'].median().reindex(loc_count.index)})
loc_median=loc_median.loc[loc_median.n>=100].sort_values('median',ascending=False).head(10)
ax[2].barh(loc_median.index[::-1],loc_median['median'].values[::-1]/1e9,color='#64748b');ax[2].set(title='10 lokasi median tertinggi (n≥100)',xlabel='Median harga (miliar Rp)')
plt.tight_layout();plt.savefig(OUT/'02_eda_distribusi_lokasi.png',dpi=150,bbox_inches='tight');plt.show()

pricedata=pd.DataFrame({'harga':y,'luas_tanah':X_clean['luas_tanah'],'luas_bangunan':X_clean['luas_bangunan'],'lokasi':X_clean['lokasi']})
rho_lt=pricedata[['harga','luas_tanah']].corr(method='spearman').iloc[0,1]
rho_lb=pricedata[['harga','luas_bangunan']].corr(method='spearman').iloc[0,1]
ratio=float(y.quantile(.95)/y.median())
print('INSIGHT 1 (target): median Rp{:,.0f}; P95 Rp{:,.0f}; rasio P95/median {:.2f}. Distribusi miring → uji log1p(target).'.format(y.median(),y.quantile(.95),ratio))
print('INSIGHT 2 (target vs fisik): Spearman Harga–LT {:.3f}, Harga–LB {:.3f}. Pertahankan luas; uji model nonlinear.'.format(rho_lt,rho_lb))
print('INSIGHT 3 (target vs lokasi): median harga di lokasi dengan ≥100 listing bervariasi dari Rp{:,.0f} hingga Rp{:,.0f}; lokasi perlu encoding kategorikal.'.format(loc_median['median'].min(),loc_median['median'].max()))
print('INSIGHT 4 (kualitas): LT yang berhasil dipulihkan dari kolom lain:',int(example.sum()),'baris; pembacaan label LT/LB penting.')

## 2. Pembagian data final (sebelum training)

Pembagian harga melalui 10 bin kuantil supaya rentang harga ekstrem relatif terwakili pada tiap bagian. Tidak ada penggunaan informasi test untuk fit, tuning, pemilihan feature atau penentuan model. **Deduplikasi sudah dilakukan sebelum pembagian** untuk mengurangi listing identik muncul pada training dan test sekaligus. Nomor indeks digunakan untuk mempertahankan baris yang sama di semua eksperimen.

In [ ]:
def stratify_bins(series):
    return pd.qcut(np.log1p(series),q=10,duplicates='drop',labels=False)
indices=np.arange(len(scope))
idx_pool,idx_test=train_test_split(indices,test_size=.15,random_state=RANDOM_STATE,stratify=stratify_bins(y))
idx_train,idx_val=train_test_split(idx_pool,test_size=.15/.85,random_state=RANDOM_STATE,stratify=stratify_bins(y.iloc[idx_pool]))
y_train,y_val,y_test=(y.iloc[a] for a in (idx_train,idx_val,idx_test))
print('Split:',{name:len(ind) for name,ind in [('train',idx_train),('validation',idx_val),('test',idx_test)]})
print('Test set akan digunakan hanya pada tahapan terakhir.')
assert len(set(idx_train)&set(idx_val))==len(set(idx_train)&set(idx_test))==len(set(idx_val)&set(idx_test))==0

## 3. Direct Modeling — 5 model regresi sebelum preprocessing lanjutan

**Alasan pemilihan model:**

1. **Linear Regression**: baseline hubungan linear dengan harga.
2. **Ridge Regression**: regularisasi untuk mengurangi ketidakstabilan koefisien.
3. **Decision Tree**: mampu menangkap aturan dan non-linearitas luas, kamar, lokasi.
4. **Random Forest (ensemble)**: agregasi pohon untuk mengurangi varians.
5. **HistGradientBoosting (ensemble)**: boosting untuk relasi kompleks harga.

Untuk kelima model, preprocessing **minimum kompatibilitas** berupa parsing dasar numerik, imputasi dan encoding kategori dilakukan dalam pipeline agar model bisa berjalan. Ini **bukan** klaim bahwa kita melatih estimator langsung pada objek teks. Target dilatih pada skala `log1p(Harga)` dan dikembalikan ke rupiah untuk evaluasi; kebijakan ini diterapkan seragam pada semua eksperimen agar pembanding adil.

In [ ]:
MODELS={
 'Linear Regression':LinearRegression(),
 'Ridge Regression':Ridge(alpha=10.0),
 'Decision Tree':DecisionTreeRegressor(max_depth=12,min_samples_leaf=8,random_state=RANDOM_STATE),
 'Random Forest':RandomForestRegressor(n_estimators=75,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=RANDOM_STATE),
 'HistGradientBoosting':HistGradientBoostingRegressor(max_iter=110,learning_rate=.07,max_leaf_nodes=25,random_state=RANDOM_STATE)
}

def make_pipeline(estimator,advanced=False,selection=False):
    numeric=NUMERIC_FULL if advanced else NUMERIC_BASE
    numeric_pipe=Pipeline([
      ('impute',SimpleImputer(strategy='median',add_indicator=True)),
      ('scale',QuantileTransformer(n_quantiles=300,output_distribution='normal',random_state=RANDOM_STATE) if advanced else StandardScaler())
    ])
    cat_pipe=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                       ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    ct=ColumnTransformer([('num',numeric_pipe,numeric),('cat',cat_pipe,CAT)],remainder='drop',sparse_threshold=0)
    steps=[('prep',ct)]
    if selection:
        steps.append(('selection',SelectFromModel(ExtraTreesRegressor(n_estimators=35,max_depth=10,min_samples_leaf=3,n_jobs=-1,random_state=RANDOM_STATE),threshold='median')))
    steps.append(('model',clone(estimator)))
    return TransformedTargetRegressor(regressor=Pipeline(steps),func=np.log1p,inverse_func=np.expm1,check_inverse=False)

def eval_metrics(y_true,pred):
    pred=np.asarray(pred,dtype=float)
    if not np.isfinite(pred).all(): raise ValueError('Prediksi tidak finite')
    return dict(MAE_Rp=mean_absolute_error(y_true,pred),MAPE_Pct=mean_absolute_percentage_error(y_true,pred)*100,R2=r2_score(y_true,pred))

records=[]
trained={}
def compare(stage,feature_set,advanced=False,selection=False):
    X_tr=feature_set.iloc[idx_train]; X_va=feature_set.iloc[idx_val]
    for name,est in MODELS.items():
        print(f'[{stage}] fitting {name} ...')
        try:
            m=make_pipeline(est,advanced=advanced,selection=selection)
            m.fit(X_tr,y_train)
            scores=eval_metrics(y_val,m.predict(X_va))
            record={'Tahap':stage,'Model':name,**scores}
            records.append(record);trained[(stage,name)]=m
        except Exception as exc:
            print('GAGAL:',repr(exc));raise
    return pd.DataFrame([r for r in records if r['Tahap']==stage]).sort_values('MAE_Rp')

baseline_results=compare('Direct (minimum)',X_basic,advanced=False)
display(baseline_results.style.format({'MAE_Rp':'Rp{:,.0f}','MAPE_Pct':'{:.2f}%','R2':'{:.4f}'}))

## 4. Modeling dengan Preprocessing

Gunakan **lima algoritma yang sama**, indeks train/validation yang sama, tetapi fiturnya telah diperbaiki (pemulihan LT/LB, validasi rentang, dua fitur turunan), imputasi dengan indikator missing dan `QuantileTransformer`. Setiap operasi imputer/scaler/encoder di-*fit* dari training saja.

In [ ]:
processed_results=compare('Preprocessing',X_clean,advanced=True)
display(processed_results.style.format({'MAE_Rp':'Rp{:,.0f}','MAPE_Pct':'{:.2f}%','R2':'{:.4f}'}))

## 5. Feature Selection (training-only)

`SelectFromModel(ExtraTreesRegressor)` memilih fitur hasil encoding dengan importansi di atas median **berdasarkan training saja**. Kelima algoritma dijalankan ulang untuk membuktikan apakah reduksi fitur membantu. Tidak ada pemilihan fitur berdasarkan data test.

In [ ]:
selected_results=compare('Feature Selection',X_clean,advanced=True,selection=True)
display(selected_results.style.format({'MAE_Rp':'Rp{:,.0f}','MAPE_Pct':'{:.2f}%','R2':'{:.4f}'}))

winner_fs=selected_results.iloc[0]['Model']
selector_model=trained[('Feature Selection',winner_fs)].regressor_
mask=selector_model.named_steps['selection'].get_support()
feature_names=selector_model.named_steps['prep'].get_feature_names_out()
chosen_features=np.asarray(feature_names)[mask]
print('Contoh fitur terpilih ({}/{}):'.format(len(chosen_features),len(feature_names)))
print(chosen_features[:40].tolist())

## 6. Hyperparameter Tuning + Cross-Validation (train saja)

Pilih kandidat **terbaik berdasarkan validation MAE** dari keluarga Preprocessing dan Feature Selection. Lakukan `RandomizedSearchCV` sebanyak maksimal enam konfigurasi × tiga fold pada **training saja**. Hasil tuning dievaluasi lagi di validation, lalu kandidat final dipilih berdasar MAE validation; MAPE dan R² dicatat sebagai pemeriksaan tambahan. Test set tetap tertutup sampai setelah model final dipilih.

In [ ]:
comparison=pd.DataFrame(records)
compare_pre=comparison[comparison['Tahap'].isin(['Preprocessing','Feature Selection'])].sort_values('MAE_Rp')
best_row=compare_pre.iloc[0]
best_stage,best_name=str(best_row['Tahap']),str(best_row['Model'])
params={
 'Linear Regression':{'regressor__model__fit_intercept':[True,False]},
 'Ridge Regression':{'regressor__model__alpha':[.1,1.,5.,10.,30.,100.]},
 'Decision Tree':{'regressor__model__max_depth':[6,10,15,22,None],'regressor__model__min_samples_leaf':[2,5,10,20]},
 'Random Forest':{'regressor__model__n_estimators':[60,100,140],'regressor__model__max_depth':[10,16,24,None],'regressor__model__min_samples_leaf':[1,2,4]},
 'HistGradientBoosting':{'regressor__model__max_iter':[80,120,160],'regressor__model__learning_rate':[.04,.07,.12],'regressor__model__max_leaf_nodes':[15,25,40]}
}
print('Kandidat tuning:',best_stage,best_name)
base=make_pipeline(MODELS[best_name],advanced=True,selection=(best_stage=='Feature Selection'))
search=RandomizedSearchCV(
  estimator=base,param_distributions=params[best_name],n_iter=2 if best_name=='Linear Regression' else 6,
  scoring='neg_mean_absolute_error',cv=KFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE),
  random_state=RANDOM_STATE,n_jobs=1,refit=True,verbose=0,error_score='raise')
search.fit(X_clean.iloc[idx_train],y_train)
print('Tuning: best CV MAE = Rp{:,.0f}'.format(-search.best_score_))
print('Parameter:',search.best_params_)
tuned=search.best_estimator_
tuned_scores=eval_metrics(y_val,tuned.predict(X_clean.iloc[idx_val]))
records.append({'Tahap':'Tuning','Model':best_name,**tuned_scores})
trained[('Tuning',best_name)]=tuned
results=pd.DataFrame(records)
results['MAE_Juta_Rp']=results['MAE_Rp']/1e6
results=results.sort_values('MAE_Rp')
display(results[['Tahap','Model','MAE_Juta_Rp','MAPE_Pct','R2']].style.format({'MAE_Juta_Rp':'{:,.1f}','MAPE_Pct':'{:.2f}','R2':'{:.4f}'}))
results.to_csv(OUT/'03_perbandingan_validation.csv',index=False)

fig,ax=plt.subplots(figsize=(12,6))
plot=results.sort_values('MAE_Juta_Rp',ascending=False)
ax.barh(plot['Tahap']+' • '+plot['Model'],plot['MAE_Juta_Rp'],color='#2563eb')
ax.set(xlabel='MAE Validation (juta Rp; lebih rendah lebih baik)',title='Perbandingan metode pada validation set')
plt.tight_layout();plt.savefig(OUT/'04_perbandingan_model.png',dpi=150,bbox_inches='tight');plt.show()

## 7. Finalisasi dan evaluasi **test set hanya satu kali**

Pilih pemenang **di antara tahap Preprocessing / Feature Selection / Tuning** berdasarkan MAE validation (Direct Modeling tetap sebagai kontrol). Kemudian buka test set **sekali** untuk melaporkan MAE, MAPE, dan R². Prediksi harga tidak menjamin akurasi karena dataset merupakan *listing/asking prices*, bukan harga transaksi dan tidak mencakup semua karakteristik rumah.

In [ ]:
final_candidates=results[results['Tahap'].isin(['Preprocessing','Feature Selection','Tuning'])]
win=final_candidates.iloc[0]
final_stage,final_name=str(win['Tahap']),str(win['Model'])
final_model=trained[(final_stage,final_name)]
# Semua kandidat final yang sah memakai X_clean; test belum dipakai di tahap mana pun.
pred_test=final_model.predict(X_clean.iloc[idx_test])
metric_test=eval_metrics(y_test,pred_test)
print('MODEL FINAL:',final_stage,'/',final_name)
print('VALIDATION:',{k:round(win[k],4) for k in ['MAE_Rp','MAPE_Pct','R2']})
print('TEST:',{k:round(v,4) for k,v in metric_test.items()})
assert metric_test['MAE_Rp']>=0 and np.isfinite(list(metric_test.values())).all()

plt.figure(figsize=(7,6))
plt.scatter(y_test/1e9, pred_test/1e9, s=8,alpha=.18,color='#2563eb')
line_max=float(np.nanpercentile(np.r_[y_test.values,pred_test],99.5)/1e9)
plt.plot([0,line_max],[0,line_max],'--',color='#ef4444',label='Prediksi sempurna')
plt.xlim(0,line_max);plt.ylim(0,line_max)
plt.xlabel('Harga aktual (miliar Rp)');plt.ylabel('Prediksi (miliar Rp)');plt.title('Aktual vs prediksi — test set');plt.legend();plt.tight_layout()
plt.savefig(OUT/'05_prediksi_vs_aktual.png',dpi=150,bbox_inches='tight');plt.show()

# Simpan pipeline scikit-learn yang sama dipakai di aplikasi Streamlit.
metadata={
 'model':final_model,'fitur_model':NUMERIC_FULL+CAT,'lokasi':sorted(X_clean['lokasi'].unique().tolist()),
 'metrik_test':metric_test,'metrik_validation':{k:float(win[k]) for k in ['MAE_Rp','MAPE_Pct','R2']},
 'model_name':final_name,'stage':final_stage,'batas_studi':'harga 100 juta-50 miliar; kamar 1-15',
 'data_rows_raw':len(raw),'data_rows_unique_scope':len(scope),
 'source':'Raw-Final-Rumah.csv','split_random_state':RANDOM_STATE
}
joblib.dump(metadata,OUT/'house_price_model.joblib',compress=3)
with (OUT/'06_ringkasan_metrik.json').open('w',encoding='utf-8') as f:
    json.dump({k:v for k,v in metadata.items() if k!='model'},f,ensure_ascii=False,indent=2)
print('Model deployment tersimpan:',OUT/'house_price_model.joblib')

## 8. Menyiapkan deployment Streamlit dan laporan

Setelah notebook selesai, unduh **ZIP hasil eksperimen** yang berisi model `.joblib`, grafik, hasil audit, file kode aplikasi, `requirements.txt`, serta ringkasan. Untuk Streamlit Cloud: taruh `app.py`, `house_price_model.joblib`, dan `requirements.txt` di root repository GitHub; lalu atur entrypoint `app.py`. **Jangan langsung mengklaim deployment sudah selesai** sebelum URL Streamlit publik dibuat dan diuji.

In [ ]:
# Buat file deployment, arsipkan dengan model terlatih dan grafik.
app_source_text = 'import joblib\nimport pandas as pd\nimport streamlit as st\nfrom pathlib import Path\n\nst.set_page_config(page_title="Prediksi Harga Rumah Depok", page_icon="🏠",layout="centered")\nst.title("🏠 Prediksi Harga Rumah Depok")\nst.caption("Estimasi berbasis listing rumah dan karakteristik fisik/lokasi. Bukan appraisal resmi atau jaminan harga transaksi.")\nMODEL_PATH=Path(__file__).parent/\'house_price_model.joblib\'\nif not MODEL_PATH.exists():\n    st.error("Model belum tersedia. Jalankan notebook sampai ekspor house_price_model.joblib, lalu letakkan berkas itu di folder yang sama dengan app.py.")\n    st.stop()\n@st.cache_resource\ndef get_artifact():return joblib.load(MODEL_PATH)\na=get_artifact()\nwith st.form(\'rumah\'):\n    lokasi=st.selectbox(\'Lokasi sebagaimana tertulis pada dataset\',options=a[\'lokasi\'])\n    kamar=st.number_input(\'Kamar tidur\',min_value=1,max_value=15,value=3)\n    mandi=st.number_input(\'Kamar mandi\',min_value=0,max_value=15,value=2)\n    garasi=st.number_input(\'Garasi / kapasitas parkir\',min_value=0,max_value=12,value=1)\n    lt=st.number_input(\'Luas tanah (m²)\',min_value=1.0,max_value=20000.0,value=90.0,step=1.0)\n    lb=st.number_input(\'Luas bangunan (m²)\',min_value=1.0,max_value=10000.0,value=70.0,step=1.0)\n    submit=st.form_submit_button(\'Prediksi harga\',type=\'primary\')\nif submit:\n    row=pd.DataFrame([{\'kamar_tidur\':float(kamar),\'kamar_mandi\':float(mandi),\'garasi\':float(garasi),\n                       \'luas_tanah\':lt,\'luas_bangunan\':lb,\'total_luas\':lt+lb,\'rasio_lb_lt\':min(lb/lt,20.),\'lokasi\':lokasi}])\n    predicted=float(a[\'model\'].predict(row)[0])\n    st.metric(\'Estimasi harga listing\',f\'Rp {predicted:,.0f}\'.replace(\',\',\'.\'))\n    st.info(f"Model: {a.get(\'model_name\',\'-\')} ({a.get(\'stage\',\'-\')}); MAE test: Rp {a[\'metrik_test\'][\'MAE_Rp\']/1e6:,.1f} juta. Ini rata-rata galat absolut pengujian, bukan interval prediksi.")\nst.divider()\nst.markdown(\'**Batasan:** model dilatih pada listing berharga sekitar Rp100 juta–Rp50 miliar dan rumah dengan 1–15 kamar. Label lokasi masih mengikuti teks sumber, sehingga tidak selalu berupa kecamatan administratif resmi. Data ini merepresentasikan harga penawaran.\')\n'
requirements_text = 'streamlit>=1.35,<2\nscikit-learn>=1.3,<2\npandas>=2,<3\nnumpy>=1.26,<3\njoblib>=1.3,<2\n'
(OUT/'app.py').write_text(app_source_text,encoding='utf-8')
(OUT/'requirements.txt').write_text(requirements_text,encoding='utf-8')
archive=shutil.make_archive('EarlyBird_Hasil_Model_dan_Deployment','zip',OUT)
print('Download hasil:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Bukan Colab: arsip sudah tersedia di path di atas.')

## 9. Pembahasan dan rangkuman untuk presentasi (wajib baca)

**Pertanyaan yang mungkin muncul saat presentasi:**

1. **Mengapa dibatasi scope?** Agar tidak mencampur target harga rumah tinggal dengan tanah, gedung komersial, atau harga per m²; batas numerik tetap perlu diuji sensitivitasnya.
2. **Kenapa direct modeling tetap ada parsing?** Karena estimator sklearn memerlukan fitur numerik; decoding satuan adalah persiapan minimum, bukan pembersihan lanjutan yang sedang diuji.
3. **Bagaimana menghindari leakage?** Deduplikasi sebelum split; satu pembagian train/validation/test; preprocessing dan selection berada di pipeline; CV pada training; test hanya pada model terpilih.
4. **MAE, MAPE, R²?** MAE = rata-rata selisih harga absolut dalam Rupiah; MAPE = rata-rata selisih persentase absolut (sensitif harga rendah); R² = seberapa besar variasi target relatif terhadap baseline rata-rata yang mampu dijelaskan oleh model.
5. **Mengapa pakai log1p harga?** Karena distribusi harga skew ke kanan; model dilatih pada log harga, namun evaluasi menggunakan skala Rupiah setelah inverse transform.
6. **Mengapa bukan `Page` sebagai fitur?** `Page` menunjukkan halaman scraping, bukan sifat rumah; berpotensi mencerminkan urutan listing daripada nilai properti.
7. **Apa kelemahan?** Tidak ada alamat rinci, ID unik, umur rumah, kondisi properti, status sertifikat, harga transaksi dan tanggal scraping; hasil bukan appraisal legal.

**Sebelum submit:** Pastikan notebook berjalan sampai akhir pada **40.200 baris asli**, semua hasil validation/test muncul, model `.joblib` berhasil dibuat, URL Streamlit publik berfungsi, dan PDF pengumpulan memuat nama, NIM, link GitHub, Streamlit, serta Colab. Jangan memasukkan angka evaluasi yang belum benar-benar dihitung.